# 03 - Data Integration (Integração de Dados)

**Etapa CRISP-DM**: Data Integration

**Objetivo**: Adquirir dados de múltiplas fontes (educacionais e socioeconômicas) e integrá-los em um dataset unificado para análise.

**Período de Dados**: 2018-2022  
**Unidade Geográfica**: 27 UFs do Brasil

---

## Contexto

Nesta etapa, vamos:

1. **Adquirir** dados de múltiplas fontes:
   - Base dos Dados (basedosdados.org)
   - SIDRA (IBGE API)
   - Processo manual de download

2. **Integrar** dados educacionais com socioeconômicos:
   - Evasão e Repetência (INEP)
   - IDH (PNUD)
   - Desemprego (IBGE)
   - PIB (IBGE)

3. **Consolidar** em um dataset único com validação de integridade

O resultado será um dataset pronto para exploração e modelagem.

## Importações e Setup

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

print('✅ Importações completas.')
print('\nNota: Este notebook usa APENAS download manual de dados.')
print('Os arquivos CSV devem estar em: data/Raw/')


✅ Importações completas.

Nota: Este notebook usa APENAS download manual de dados.
Os arquivos CSV devem estar em: data/Raw/


---

## Seção 1: Aquisição de Dados de Múltiplas Fontes

**Objetivo**: Documentar e demonstrar diferentes estratégias para adquirir dados educacionais e socioeconômicos.

Exploramos três abordagens:
1. **Base dos Dados** (API - requer autenticação Google Cloud)
2. **SIDRA** (API do IBGE - acesso gratuito)
3. **Download Manual** (fallback quando APIs não funcionam)

### Abordagem 1: Base dos Dados (basedosdados.org)

A plataforma Base dos Dados oferece dados públicos brasileiros através de uma API.

**Requisitos**:
- Instalar: `pip install basedosdados`
- Autenticação: Requer Google Cloud com créditos
- Dados disponíveis: Diversos indicadores educacionais

**Vantagens**: Dados estruturados, API bem documentada

**Desvantagens**: Requer autenticação, pode ter custos

### Abordagem 2: SIDRA (API do IBGE)

SIDRA é o Sistema Integrado de Dados Agregados do IBGE.

**Requisitos**:
- Instalar: `pip install sidra`
- Autenticação: Não requer
- Dados disponíveis: Desemprego, PIB, população, etc.

**Vantagens**: Gratuito, acesso público, dados confiáveis

**Desvantagens**: API pode ser lenta, dados em português

### Abordagem 3: Download Manual (Fallback)

Quando as APIs não funcionam, fazemos download manual de arquivos CSV/Excel.

**Processo**:
1. Acessar plataforma (SIDRA, PNUD, etc.)
2. Selecionar período (2018-2022)
3. Selecionar unidades (27 UFs)
4. Baixar arquivo CSV
5. Salvar em `data/Raw/`

**Vantagens**: Sempre funciona, sem dependências

**Desvantagens**: Manual, propenso a erros

In [3]:
print("\n" + "="*70)
print("VERIFICANDO DADOS - DOWNLOAD MANUAL")
print("="*70)

data_dir = '../data/Raw/'

print(f"\nProcurando arquivos CSV em: {data_dir}")

dados_disponiveis = {}
for arquivo in os.listdir(data_dir):
    if arquivo.endswith('.csv'):
        caminho = os.path.join(data_dir, arquivo)
        try:
            df_temp = pd.read_csv(caminho, nrows=1)
            dados_disponiveis[arquivo] = caminho
            print(f"  ✅ Encontrado: {arquivo}")
        except:
            print(f"  ❌ Erro ao ler: {arquivo}")

if not dados_disponiveis:
    print("\n❌ Nenhum arquivo CSV encontrado em data/Raw/")
    print("\nInstruções:")
    print("  1. Baixe os arquivos manualmente")
    print("  2. Coloque em: data/Raw/")
    print("  3. Arquivos esperados:")
    print("     - dados_educacao.csv")
    print("     - dados_idh.csv")
    print("     - dados_pib.csv")
    print("     - dados_desemprego.csv")
else:
    print(f"\n✅ Total: {len(dados_disponiveis)} arquivo(s) encontrado(s)")



VERIFICANDO DADOS - DOWNLOAD MANUAL

Procurando arquivos CSV em: ../data/Raw/
  ✅ Encontrado: RELATORIO_DTB_BRASIL_2024_DISTRITOS.csv
  ✅ Encontrado: desemprego_sindra.csv
  ✅ Encontrado: nascidos_vivos_total_sidra.csv
  ✅ Encontrado: Deslocamento.csv
  ✅ Encontrado: nascidos_vivos_adolescentes_sidra.csv
  ✅ Encontrado: gini_sidra.csv
  ✅ Encontrado: renda_sintra.csv
  ✅ Encontrado: pib_sidra.csv

✅ Total: 8 arquivo(s) encontrado(s)


---

## Seção 2: Combinação de Dados Socioeconômicos

**Objetivo**: Mesclar dados educacionais com indicadores socioeconômicos.

Nesta seção:
- Carregamos dados de múltiplas fontes
- Padronizamos nomes de colunas e UFs
- Fazemos merge por UF e Ano
- Validamos integridade
- Produzimos dataset final integrado

In [5]:
print("="*70)
print("COMBINAÇÃO DE DADOS - EDUCAÇÃO + SOCIOECONÔMICOS")
print("="*70)

# Diretório de dados processados
data_dir = '../data/Processed'

# Verificar arquivos disponíveis
print("\nVerificando dados disponíveis em data/Processed/...\n")

dados_disponiveis = {}
for arquivo in os.listdir(data_dir):
    if arquivo.endswith('.csv'):
        caminho = os.path.join(data_dir, arquivo)
        try:
            df_temp = pd.read_csv(caminho, nrows=1)  # Carregar apenas header
            dados_disponiveis[arquivo] = caminho
            print(f"  Encontrado: {arquivo}")
        except:
            pass

if not dados_disponiveis:
    print("  Nenhum arquivo CSV encontrado em data/Processed/")
    print("  Use o notebook 02_Preparacao_Dados.ipynb primeiro")

COMBINAÇÃO DE DADOS - EDUCAÇÃO + SOCIOECONÔMICOS

Verificando dados disponíveis em data/Processed/...

  Encontrado: Saneamento.csv
  Encontrado: indicesEnsino.csv
  Encontrado: gravidez_adolescente_2018_2022.csv
  Encontrado: gini_2018_2022.csv
  Encontrado: dados_modelo_final.csv
  Encontrado: renda_2018_2022.csv
  Encontrado: indicadores_educacionais_2018_2022.csv
  Encontrado: pib_2018_2022.csv
  Encontrado: desemprego_2018_2022.csv
  Encontrado: dados_finais_analise.csv
  Encontrado: Deslocamento_Enriquecido_com_UF_e_Codigos.csv
  Encontrado: idhm_2018_2022.csv


In [6]:
# Se houver arquivo consolidado, carregar direto
if 'dados_modelo_final.csv' in dados_disponiveis:
    print("\nCarregando dataset já consolidado...\n")
    df_final = pd.read_csv(os.path.join(data_dir, 'dados_modelo_final.csv'))
    print(f"Dataset consolidado:")
    print(f"  Registros: {len(df_final)}")
    print(f"  Colunas: {len(df_final.columns)}")
    print(f"  Período: {df_final['Ano'].min()}-{df_final['Ano'].max()}")
    print(f"\nColunas disponíveis:")
    for col in df_final.columns:
        print(f"  - {col}")
else:
    print("\nDataset final ainda não foi criado.")
    print("Execute o processo de combinação:")
    print("  1. Carregue dados educacionais (02_Preparacao_Dados.ipynb)")
    print("  2. Carregue dados socioeconômicos (csv em data/Raw/)")
    print("  3. Faça merge por UF e Ano")
    print("  4. Valide e salve como dados_modelo_final.csv")


Carregando dataset já consolidado...

Dataset consolidado:
  Registros: 135
  Colunas: 17
  Período: 2018-2022

Colunas disponíveis:
  - UF
  - Ano
  - Taxa_Abandono_Media
  - Taxa_Abandono_EF
  - Taxa_Abandono_EM
  - Taxa_Reprovacao_Media
  - Taxa_Reprovacao_EF
  - Taxa_Reprovacao_EM
  - IDHM
  - Taxa_Desemprego
  - Renda_Per_Capita
  - Indice_Gini_x
  - Taxa_Gravidez_Adolescente_x
  - PIB_Total_MilReais_x
  - Indice_Gini_y
  - Taxa_Gravidez_Adolescente_y
  - PIB_Total_MilReais_y


In [7]:
# Se carregou com sucesso, mostrar amostra
if 'df_final' in locals() and df_final is not None:
    print("\nAmostra dos dados consolidados:")
    print(df_final.head(10))
    
    print("\nEstatísticas descritivas:")
    print(df_final.describe())


Amostra dos dados consolidados:
          UF   Ano  Taxa_Abandono_Media  Taxa_Abandono_EF  Taxa_Abandono_EM  \
0   Rondônia  2018                 2.20               1.4               3.0   
1       Acre  2018                 2.95               2.2               3.7   
2   Amazonas  2018                 3.85               2.8               4.9   
3    Roraima  2018                 2.90               2.0               3.8   
4       Pará  2018                 4.60               3.6               5.6   
5      Amapá  2018                 3.35               2.4               4.3   
6  Tocantins  2018                 1.70               1.1               2.3   
7   Maranhão  2018                 3.10               2.3               3.9   
8      Piauí  2018                 2.40               1.8               3.0   
9      Ceará  2018                 1.05               0.9               1.2   

   Taxa_Reprovacao_Media  Taxa_Reprovacao_EF  Taxa_Reprovacao_EM   IDHM  \
0                   6.

In [8]:
# Validação de integridade
if 'df_final' in locals() and df_final is not None:
    print("\n" + "="*70)
    print("VALIDAÇÃO DE INTEGRIDADE")
    print("="*70)
    
    # Valores faltantes
    print("\nValores faltantes por coluna:")
    missing = df_final.isnull().sum()
    if missing.sum() > 0:
        print(missing[missing > 0])
    else:
        print("  Nenhum valor faltante")
    
    # Distribuição por UF e Ano
    print("\nRegistros por Ano:")
    print(df_final['Ano'].value_counts().sort_index())
    
    # Tipos de dados
    print("\nTipos de dados:")
    print(df_final.dtypes)


VALIDAÇÃO DE INTEGRIDADE

Valores faltantes por coluna:
  Nenhum valor faltante

Registros por Ano:
Ano
2018    27
2019    27
2020    27
2021    27
2022    27
Name: count, dtype: int64

Tipos de dados:
UF                              object
Ano                              int64
Taxa_Abandono_Media            float64
Taxa_Abandono_EF               float64
Taxa_Abandono_EM               float64
Taxa_Reprovacao_Media          float64
Taxa_Reprovacao_EF             float64
Taxa_Reprovacao_EM             float64
IDHM                           float64
Taxa_Desemprego                float64
Renda_Per_Capita               float64
Indice_Gini_x                  float64
Taxa_Gravidez_Adolescente_x    float64
PIB_Total_MilReais_x           float64
Indice_Gini_y                  float64
Taxa_Gravidez_Adolescente_y    float64
PIB_Total_MilReais_y           float64
dtype: object


---

## Conclusões

Nesta etapa de **Data Integration** realizamos:

1. **Avaliação** de múltiplas estratégias de aquisição de dados:
   - Base dos Dados API (complexo, requer autenticação)
   - SIDRA API (gratuito, fácil)
   - Download manual (fallback confiável)

2. **Integração** de dados educacionais com socioeconômicos:
   - Merge por UF e Ano
   - Padronização de nomes
   - Tratamento de valores faltantes

3. **Validação** de integridade do dataset final

### Dataset Integrado

O resultado é um dataset com:
- **Período**: 2018-2022
- **Unidades**: 27 UFs
- **Registros**: 135 (27 UFs × 5 anos)
- **Targets**: Taxa de Abandono, Taxa de Reprovação
- **Features**: IDHM, Desemprego, Renda Per Capita, Gini, Gravidez Adolescente, PIB

### Próximas Etapas

O dataset integrado será utilizado em:
- **Modeling**: Treinamento e otimização de modelos preditivos
- **Evaluation**: Avaliação de performance
- **Deployment**: Dashboard e visualizações

### Limitações e Considerações

- Qualidade dos dados depende de fontes oficiais (INEP, IBGE, PNUD)
- Alguns anos podem ter dados faltantes em certos estados
- APIs podem estar indisponíveis ou com limitações
- Validação manual pode ser necessária para outliers
- Correlação entre variáveis não implica causalidade